In [1]:
import cv2
import os

# Paths
input_dir = "extraction"  # Contains Open and Closed folders
output_dir = "eye_dataset"

# Haar Cascade files
face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)

eye_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_eye.xml"
)

# Create output folders
classes = ["Open", "Closed"]

for cls in classes:
    os.makedirs(os.path.join(output_dir, cls), exist_ok=True)

for cls in classes:

    input_class_dir = os.path.join(input_dir, cls)
    output_class_dir = os.path.join(output_dir, cls)

    count = 0

    for filename in os.listdir(input_class_dir):

        img_path = os.path.join(input_class_dir, filename)

        img = cv2.imread(img_path)

        if img is None:
            continue

        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

        # Detect face
        faces = face_cascade.detectMultiScale(
            gray,
            scaleFactor=1.1,
            minNeighbors=5,
            minSize=(50, 50)
        )

        for (x, y, w, h) in faces:

            face_roi_gray = gray[y:y+h, x:x+w]
            face_roi_color = img[y:y+h, x:x+w]

            # Detect eyes within face
            eyes = eye_cascade.detectMultiScale(
                face_roi_gray,
                scaleFactor=1.1,
                minNeighbors=8
            )

            for i, (ex, ey, ew, eh) in enumerate(eyes):

                eye_crop = face_roi_color[
                    ey:ey+eh,
                    ex:ex+ew
                ]

                save_path = os.path.join(
                    output_class_dir,
                    f"{count}_{i}.jpg"
                )

                cv2.imwrite(save_path, eye_crop)

            count += 1

    print(f"{cls}: {count} images processed")

print("Eye extraction completed!")

Open: 260 images processed
Closed: 175 images processed
Eye extraction completed!
